In [ ]:
# Connect to source data
!gdown --id 1r_V2sb5ptKvzrGv-afRFZlRutewTDFjE
!unzip -q super-ai-engineer-ss-6-word-segmentation.zip

In [ ]:
!tar -xf /content/Corpus-LST20.tar.gz -C /content/
!ls /content/

In [ ]:
import os
import glob

# ⚠️ เปลี่ยนชื่อโฟลเดอร์ LST20_Corpus ให้ตรงกับที่แตกไฟล์มาได้จาก Step 1 นะครับ
TRAIN_DIR = "/content/LST20_Corpus/train"

def load_and_convert_lst20(folder_path):
    char_data = []

    # วิ่งหาไฟล์ .txt ทั้งหมดในโฟลเดอร์
    for filepath in glob.glob(os.path.join(folder_path, '*.txt')):
        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()

                # ข้ามบรรทัดว่าง (ตัวแบ่งประโยค)
                if not line:
                    continue

                # แยกข้อมูลด้วย Tab (\t) คอลัมน์แรก [0] คือคำ (Word)
                parts = line.split('\t')
                word = parts[0]

                # กฎของ LST20: ช่องว่างมักจะถูกแทนด้วยสัญลักษณ์ '_' (Underscore)
                # และโจทย์บอกว่าช่องว่างไม่ถูกนำไปคิดในไฟล์ Submit เราจึงข้ามมันไป
                if word == '_' or word.strip() == '':
                    continue

                # แตกคำเป็นตัวอักษร + แปะป้าย BIE
                word_len = len(word)
                for i, char in enumerate(word):
                    if word_len == 1:
                        tag = 'B_WORD'
                    elif i == 0:
                        tag = 'B_WORD'
                    elif i == word_len - 1:
                        tag = 'E_WORD'
                    else:
                        tag = 'I_WORD'

                    char_data.append((char, tag))

    return char_data

# ลองรันเพื่อโหลดและแปลงข้อมูล Train
print("Loading and processing LST20 Train data...")
train_char_data = load_and_convert_lst20(TRAIN_DIR)

print(f"✅ ประมวลผลเสร็จสิ้น! ได้ตัวอักษรทั้งหมด: {len(train_char_data)} ตัว")
print("\n--- ตัวอย่างข้อมูล 15 ตัวอักษรแรก ---")
for char, tag in train_char_data[:15]:
    print(f"'{char}' : {tag}")

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

print("กำลังสร้างพจนานุกรมตัวอักษร (Vocabulary)...")

# 1. รวบรวมตัวอักษรทั้งหมดที่มีในข้อมูล Train
chars = set([char for char, tag in train_char_data])
char2idx = {c: i+2 for i, c in enumerate(chars)}
char2idx["PAD"] = 0 # สำหรับเติมช่องว่างให้ความยาวเท่ากัน
char2idx["UNK"] = 1 # สำหรับตัวอักษรแปลกๆ ที่โผล่มาใน Test

# 2. เตรียม Tags
tags = ["B_WORD", "I_WORD", "E_WORD", "O"]
tag2idx = {t: i for i, t in enumerate(tags)}
idx2tag = {i: t for t, i in tag2idx.items()}

# 3. 🔴 เพิ่มโค้ดส่วนนี้: หั่นข้อมูลทั้งหมดเป็นท่อนๆ ท่อนละ 100 ตัวอักษร
MAX_LEN = 100
sentences = [train_char_data[i:i + MAX_LEN] for i in range(0, len(train_char_data), MAX_LEN)]
print(f"แบ่งข้อมูลได้ทั้งหมด {len(sentences)} ท่อน")

# 4. ฟังก์ชันสำหรับแปลงตัวอักษรและ Tag ให้เป็นตัวเลข
def encode_sentences(sentences_list):
    # ถ้าหา Tag ไม่เจอ ให้มองเป็น "O" ไว้ก่อนเพื่อป้องกัน Error
    X = [[char2idx.get(char, 1) for char, tag in s] for s in sentences_list]
    y = [[tag2idx.get(tag, tag2idx["O"]) for char, tag in s] for s in sentences_list]
    return X, y

# ส่ง list ที่หั่นแล้วเข้าไปแปลง
X_idx, y_idx = encode_sentences(sentences)

# 5. Padding (เติม 0 ให้ความยาวเท่ากันทุกท่อนเป๊ะๆ 100 ตัว)
X_pad = pad_sequences(X_idx, maxlen=MAX_LEN, padding='post', value=char2idx["PAD"])
y_pad = pad_sequences(y_idx, maxlen=MAX_LEN, padding='post', value=tag2idx["O"])

# 6. แปลง Label เป็น One-Hot Encoding สำหรับเทรน Neural Network
y_cat = to_categorical(y_pad, num_classes=len(tags))

print(f"✅ เตรียมข้อมูลเสร็จสิ้น! ขนาดของ X: {X_pad.shape}, ขนาดของ y: {y_cat.shape}")

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, TimeDistributed, Dense, Dropout

print("กำลังสร้างโมเดล BiLSTM...")
model = Sequential([
    # 1. Embedding Layer: แปลงตัวเลขเป็นเวกเตอร์ความหมาย
    Embedding(input_dim=len(char2idx), output_dim=128, input_length=MAX_LEN),

    # 2. BiLSTM Layer: สมองหลักที่ใช้อ่านหน้า-หลัง (ใส่ 2 ชั้นเพื่อความโหดทะลุ 0.97)
    Bidirectional(LSTM(units=128, return_sequences=True)),
    Dropout(0.3),
    Bidirectional(LSTM(units=64, return_sequences=True)),

    # 3. Output Layer: ตัดสินใจทายป้าย B, I, E ในทุกๆ ตัวอักษร
    TimeDistributed(Dense(len(tags), activation='softmax'))
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

print("\n🚀 เริ่มต้นการเทรน Deep Learning...")
# เทรน 10 รอบ (Epochs) ถ้าใช้ GPU จะเสร็จในไม่กี่นาที
history = model.fit(X_pad, y_cat, batch_size=256, epochs=10, validation_split=0.1)

In [ ]:
import pandas as pd

# 1. อ่านไฟล์ Test และหั่นเป็นท่อนละ 100
with open('/content/ws_test.txt', 'r', encoding='utf-8') as f:
    test_text = f.read()

test_chunks = [test_text[i:i+MAX_LEN] for i in range(0, len(test_text), MAX_LEN)]

# 2. แปลงเป็นตัวเลขและ Padding
X_test_idx = [[char2idx.get(c, char2idx["UNK"]) for c in chunk] for chunk in test_chunks]
X_test_pad = pad_sequences(X_test_idx, maxlen=MAX_LEN, padding='post', value=char2idx["PAD"])

# 3. ให้โมเดลทำนาย
print("กำลังทำนายผล Test Data...")
y_pred_prob = model.predict(X_test_pad)
y_pred_idx = np.argmax(y_pred_prob, axis=-1)

# 4. แปลงกลับเป็นรายตัวอักษร และตัด Padding ทิ้งให้ความยาวเท่าต้นฉบับเป๊ะๆ
y_pred_flat = []
for i, chunk in enumerate(test_chunks):
    chunk_pred = y_pred_idx[i][:len(chunk)] # ตัดเอาเฉพาะความยาวจริงของท่อนนั้น
    y_pred_flat.extend(chunk_pred)

y_pred_tags = [idx2tag[idx] for idx in y_pred_flat]

print(f"ความยาวตัวอักษร Test ทั้งหมด: {len(test_text)}")
print(f"ความยาวผลทำนายที่ได้: {len(y_pred_tags)} (ต้องเท่ากัน)")

# 5. หยอดลงไฟล์ CSV
sub_df = pd.read_csv('/content/ws_sample_submission.csv')
sub_df['Predicted'] = sub_df['Id'].apply(lambda x: y_pred_tags[x - 1])

sub_df.to_csv('/content/submission_bilstm.csv', index=False)
print("✅ เซฟไฟล์ submission_bilstm.csv เรียบร้อย เอาไปล้ม Baseline 0.97 ได้เลย!")

In [ ]:
!pip install pythainlp attacut

In [ ]:
import pandas as pd
from pythainlp.tokenize import word_tokenize

# 1. โหลดไฟล์ Test
with open('/content/ws_test.txt', 'r', encoding='utf-8') as f:
    test_text = f.read()

# 2. ใช้ Deep Learning ระดับประเทศ (AttaCut) ตัดคำให้เลย!
print("กำลังให้ AI (AttaCut) ตัดคำ... (ใช้เวลาไม่กี่วินาที)")
# ลองใช้ engine='attacut' (Deep Learning) หรือ 'newmm' (Dictionary-based)
words = word_tokenize(test_text, engine='attacut', keep_whitespace=True)

# 3. แปลงคำที่ตัดมาได้ กลับเป็นป้าย BIE Tags
y_pred_tags = []
for word in words:
    word_len = len(word)

    # ถ้าเป็นช่องว่างหรือการขึ้นบรรทัดใหม่ เราใส่ Tag 'O' (หรืออะไรก็ได้) ไว้ก่อน
    # เพื่อให้ Index ของตัวอักษรไม่เลื่อน
    if word.strip() == '':
        y_pred_tags.extend(['O'] * word_len)
    else:
        for i in range(word_len):
            if word_len == 1:
                y_pred_tags.append('B_WORD')
            elif i == 0:
                y_pred_tags.append('B_WORD')
            elif i == word_len - 1:
                y_pred_tags.append('E_WORD')
            else:
                y_pred_tags.append('I_WORD')

print(f"ความยาวตัวอักษรในไฟล์ Test: {len(test_text)} ตัว")
print(f"ความยาวป้าย BIE ที่สร้างได้: {len(y_pred_tags)} ตัว (ต้องเท่ากันเป๊ะ!)")

# 4. หยอดคำตอบลงไฟล์ Submission
sub_df = pd.read_csv('/content/ws_sample_submission.csv')

# ดึง Tag จาก y_pred_tags โดยใช้ Id เป็น Index (อย่าลืม -1 เพราะ List เริ่มที่ 0)
sub_df['Predicted'] = sub_df['Id'].apply(lambda x: y_pred_tags[x - 1])

# 5. เซฟไฟล์ไปส่ง!
sub_df.to_csv('/content/submission_pythainlp_attacut.csv', index=False)
print("✅ เซฟไฟล์ 'submission_pythainlp_attacut.csv' เรียบร้อย เอาไปล้ม Baseline ได้เลย!")